[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/demos/S2_columnas_ridge.ipynb)

# Demo: columnas y Ridge con MiBici

Demo en vivo de S2. No se entrega. Seguimos como el equipo de análisis de MiBici: queremos predecir cuántas bicis salen
de cada estación en cada hora, con un día de anticipación, para planear la redistribución.

Tiene dos partes. La **parte 1** (inventar columnas) va con la slide *En Python: inventar columnas*; la **parte 2** (Ridge)
va con la slide *En Python: Ridge*. Corre las celdas en orden; las dos primeras preparan el entorno hasta que aparezca `✅ entorno listo`.
Todo el notebook corre en uno o dos minutos.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
# las interacciones del Lab 2, como texto: "51_8_0" es la estación 51, a las 8, entre semana
df["est_hora"] = df.station_id.astype(str) + "_" + df.hour_of_day.astype(str)
df["est_hora_finde"] = df.est_hora + "_" + df.is_weekend.astype(str)
df["est_hora_dia"] = df.est_hora + "_" + df.day_of_week.astype(str)
train = df[df.particion == "train"]       # octubre a junio: aquí se ajusta
julio = df[df.particion == "val"]         # julio: aquí se valida
E_REF = 1.95                              # la tabla por estación y hora del Lab 1, medida en julio
print(f"{df.station_id.nunique()} estaciones   train: {len(train):,} filas   julio: {len(julio):,} filas")

**¿Qué esperamos ver?** 40 estaciones, 254,440 filas de train y 29,760 de julio. Cada fila es una estación en una hora.
La referencia que hay que vencer es la tabla del Lab 1: en julio se equivoca por 1.95 bicis por hora.

# Parte 1: inventar columnas

*Se corre en la slide "En Python: inventar columnas".*

La recta solo sabe sumar columnas por pesos. Si queremos que sepa algo nuevo (qué pasó ayer, que las 8 a.m. son distintas
de las 9, que la estación 51 madruga y otra no), tenemos que dárselo como columna. Vemos tres formas de inventarlas.

## 1. Rezago: lo que pasó hace 24 y 168 horas

$$x^{(24)}_t = y_{t-24} \qquad x^{(168)}_t = y_{t-168}$$

En palabras: la columna nueva de la hora $t$ es lo que salió en esa misma estación 24 horas antes (ayer a esta hora)
o 168 horas antes (hace una semana a esta hora).

**¿Por qué 24 y 168, y no la hora anterior?** Porque planeamos con un día de anticipación: a las 8 a.m. de hoy no sabemos
lo que pasará a las 7 a.m. de mañana. Lo más reciente que podemos usar honestamente es la misma hora de ayer.

Los datos ya traen `departures_lag_24h` y `departures_lag_168h`. Comprobemos que son un simple `shift` dentro de cada estación.
Los datos están ordenados por hora, así que `groupby("station_id")` desplaza cada estación con su propia historia.

In [ ]:
df["shift_24"] = df.groupby("station_id").departures.shift(24)
df["shift_168"] = df.groupby("station_id").departures.shift(168)

e51 = df[df.station_id == 51].set_index("hour")
tabla_rezago = e51.loc["2026-07-07 05:00":"2026-07-07 14:00",
                       ["departures", "shift_24", "departures_lag_24h", "shift_168", "departures_lag_168h"]]
display(tabla_rezago.astype("Int64"))

for k in (24, 168):
    hay = df[f"shift_{k}"].notna()
    iguales = (df.loc[hay, f"shift_{k}"] == df.loc[hay, f"departures_lag_{k}h"]).mean()
    print(f"shift({k}) coincide con departures_lag_{k}h en el {iguales:.0%} de las {hay.sum():,} filas donde tiene de dónde copiar; "
          f"en las primeras {k} horas de cada estación ({(~hay).sum():,} filas) deja un hueco")

**¿Qué esperamos ver?** Diez horas del martes 7 de julio en la estación 51. La columna `shift_24` es idéntica a `departures_lag_24h`
y `shift_168` a `departures_lag_168h`. Por ejemplo, a las 8 a.m. salieron 28; ayer a esa hora, 62; hace una semana, 16.
La coincidencia es del 100 % donde `shift` tiene de dónde copiar. Al principio de cada estación (960 filas para 24 h, 6,720 para 168 h)
`shift` deja un hueco; los datos sí traen el rezago porque se calculó con historia anterior a octubre.

In [ ]:
semana = e51.loc["2026-07-06":"2026-07-12"]
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(semana.index, semana.departures, color=COLORES["tinta"], lw=1.8, label="lo que salió (departures)")
ax.plot(semana.index, semana.departures_lag_24h, color=COLORES["w"], lw=1.3, ls="--", label="ayer a esta hora (lag 24 h)")
ax.plot(semana.index, semana.departures_lag_168h, color=COLORES["lambda"], lw=1.3, ls=":", label="hace una semana (lag 168 h)")
ax.set(ylabel="salidas por hora", ylim=(-2, 100), title="Estación 51, semana del lunes 6 al domingo 12 de julio")
ax.legend(loc="upper left", ncol=3)
plt.tight_layout()
plt.show()
mae_lag = {c: np.mean(np.abs(julio[c] - julio.departures)) for c in ["departures_lag_24h", "departures_lag_168h"]}
print(f"Si usamos el rezago tal cual como predicción, en julio fallamos por {mae_lag['departures_lag_24h']:.2f} (ayer) "
      f"y {mae_lag['departures_lag_168h']:.2f} (hace una semana) bicis por hora")

**¿Qué esperamos ver?** La línea azul es la negra corrida un día a la derecha; la naranja, corrida una semana.
La de ayer falla el lunes, porque copia al domingo; la de hace una semana sí sabe que el lunes es lunes.
Como predicción sola, en julio el rezago de ayer falla por 2.50 bicis y el de hace una semana por 2.30: ninguno le gana a la tabla (1.95).
Los rezagos sirven como **columnas**, para que la recta los combine con otras.

## 2. Categoría: la hora como 24 columnas de 0 y 1

$$[x = h] = \begin{cases} 1 & \text{si la fila es de la hora } h \\ 0 & \text{si no} \end{cases} \qquad h = 0, 1, \dots, 23$$

En palabras: en lugar de una columna con el número de la hora, 24 columnas; cada fila tiene un 1 en la de su hora y 0 en las demás.

**¿Por qué no la hora como número?** Con la hora como número, la recta le pondría a cada hora un peso proporcional:
las 23 h valdrían 23 veces lo que vale la 1 a.m. Las salidas no crecen así: suben a las 8, bajan a mediodía y vuelven a subir.
Con 24 columnas, cada hora recibe su propio peso. Eso hace `OneHotEncoder`.

In [ ]:
seis = train[train.station_id == 51].set_index("hour").loc["2026-06-22", ["hour_of_day"]].iloc[[0, 3, 7, 8, 13, 18]]
uno_caliente = OneHotEncoder(sparse_output=False).fit(train[["hour_of_day"]])
matriz = pd.DataFrame(uno_caliente.transform(seis), columns=[f"h{h}" for h in range(24)],
                      index=[f"{h} h" for h in seis.hour_of_day]).astype(int)
print(f"6 filas, {matriz.shape[1]} columnas")
display(matriz)

fig, ax = plt.subplots(figsize=(10, 2.6))
ax.imshow(matriz, cmap="Blues", aspect="auto", vmin=0, vmax=1.3)
ax.set_xticks(range(24), range(24))
ax.set_yticks(range(len(matriz)), matriz.index)
ax.set(xlabel="columna (hora)", ylabel="fila", title="Un solo 1 por fila, en la columna de su hora")
plt.tight_layout()
plt.show()

**¿Qué esperamos ver?** Seis filas del lunes 22 de junio (0, 3, 7, 8, 13 y 18 h) y 24 columnas. Cada fila tiene un solo 1:
la de las 8 h lo tiene en `h8`. En el heatmap, una diagonal de cuadros azules sobre fondo blanco. Casi toda la matriz son ceros.

## 3. Interacción: estación × hora

Si sumamos la estación y la hora por separado, la recta predice

$$\hat y = w_{\text{est}} + w_{\text{hora}} + \dots$$

En palabras: cada estación sube o baja **toda la curva** del día, pero la forma de la curva (cuándo es el pico) es la misma para todas.

Con la interacción, cada pareja estación y hora recibe su propio peso:

$$\mathbb 1[\text{est} = 51 \wedge \text{hora} = 8] = 1 \text{ solo si la fila es de la estación 51 y de las 8 h}$$

En palabras: una columna por cada combinación. Así cada estación puede tener su pico a la hora que le toca.

**¿Hace falta?** Veamos si las estaciones tienen la misma forma. Tomamos 8 estaciones: las 4 con más salidas por la mañana
en proporción y las 4 con más por la tarde, entre las que mueven al menos 3 bicis por hora entre semana.

In [ ]:
entre_semana = train[train.is_weekend == 0]
curvas = entre_semana.groupby(["station_id", "hour_of_day"]).departures.mean().unstack()
movidas = curvas[curvas.mean(axis=1) >= 3]
manana_vs_tarde = movidas.loc[:, 6:9].sum(axis=1) / movidas.loc[:, 16:19].sum(axis=1)
ocho = list(manana_vs_tarde.nlargest(4).index) + list(manana_vs_tarde.nsmallest(4).index[::-1])
print("Estaciones:", ocho)
display(curvas.loc[ocho, [6, 7, 8, 9, 13, 17, 18, 19]].round(1))

# lo que puede dibujar la versión sumada: estación + hora (sin rezagos), ajustada con las mismas filas
sumada = make_pipeline(OneHotEncoder(), LinearRegression()).fit(entre_semana[["station_id", "hour_of_day"]],
                                                                 entre_semana.departures)
rejilla = pd.DataFrame([(e, h) for e in ocho for h in range(24)], columns=["station_id", "hour_of_day"])
curvas_sumada = pd.DataFrame(sumada.predict(rejilla).reshape(len(ocho), 24), index=ocho)

# cada renglón entre su máximo: así se compara la forma del día, no el tamaño de la estación
fig, (izq, der) = plt.subplots(1, 2, figsize=(13, 3.9), sharey=True)
for eje, datos, titulo in [(izq, curvas.loc[ocho], "Real: promedio de train"),
                           (der, curvas_sumada, "Lo que puede dibujar estación + hora")]:
    forma = datos.div(datos.max(axis=1), axis=0).clip(lower=0)
    im = eje.imshow(forma, cmap="Blues", aspect="auto", vmin=0, vmax=1)
    eje.set_xticks(range(0, 24, 2), range(0, 24, 2))
    eje.set(title=titulo, xlabel="hora del día")
izq.set_yticks(range(len(ocho)), [f"est. {e}" for e in ocho])
fig.colorbar(im, ax=[izq, der], label="fracción del máximo de la estación", shrink=0.9)
plt.show()

**¿Qué esperamos ver?** En la tabla, la estación 51 tiene su pico a las 8 a.m. (29 salidas) y la 194 a las 6 p.m. (48.7),
con apenas 3 a las 8 a.m. Cada renglón del heatmap está dividido entre su máximo para comparar formas.
A la izquierda, dos grupos: las estaciones 51, 58, 271 y 157 se encienden a las 7 y 8 a.m. (de ahí sale la gente a trabajar)
y las 36, 35, 65 y 194 en la tarde, de 2 a 7 p.m. (de ahí sale la gente de regreso).
A la derecha, la versión sumada les pone a **todas** la misma forma del día, con un pico a las 8 a.m. y otro a las 6 p.m.:
en la 194 pinta media mañana de azul cuando en la realidad casi no sale nadie.
Para la 194, eso es predecir bicis de más a las 8 a.m. y de menos a las 6 p.m.

### ¿Cuánto mejora en julio?

Replicamos `prueba_columnas` del Lab 2: las columnas de texto pasan por `OneHotEncoder`, las numéricas pasan tal cual,
se ajusta `LinearRegression` con train y medimos el MAE en julio:

$$\text{MAE}_{\text{julio}} = \frac{1}{m}\sum_{i \in \text{julio}} \big|\hat y^{(i)} - y^{(i)}\big|$$

En palabras: por cuántas bicis nos equivocamos en promedio en cada estación y hora de julio, sin importar el signo.

In [ ]:
CATEGORIAS = ["hour_of_day", "day_of_week", "station_id", "est_hora", "est_hora_finde", "est_hora_dia"]
intentos = {}     # se guarda cada lista ya probada: volver a escogerla en el widget es instantáneo


def prueba_columnas(columnas):
    columnas = list(columnas)
    clave = ", ".join(columnas)
    if clave not in intentos:
        cats = [col for col in columnas if col in CATEGORIAS]
        nums = [col for col in columnas if col not in CATEGORIAS]
        partes = ([("cat", OneHotEncoder(handle_unknown="ignore"), cats)] if cats else []) + \
                 ([("num", "passthrough", nums)] if nums else [])
        m = make_pipeline(ColumnTransformer(partes), LinearRegression()).fit(train[columnas], train.departures)
        intentos[clave] = np.mean(np.abs(m.predict(julio[columnas]) - julio.departures))
    return intentos[clave]


comparacion = pd.Series({
    "sumadas: rezagos + hora + estación": prueba_columnas(["departures_lag_24h", "departures_lag_168h", "hour_of_day", "station_id"]),
    "la tabla del Lab 1": E_REF,
    "interacción: rezagos + est_hora_finde": prueba_columnas(["departures_lag_24h", "departures_lag_168h", "est_hora_finde"]),
}, name="MAE en julio")
display(comparacion.round(3).to_frame())

fig, ax = plt.subplots(figsize=(8, 3.6))
colores = [COLORES["error"], COLORES["datos"], COLORES["yhat"]]
ax.barh(comparacion.index, comparacion.values, color=colores, edgecolor=COLORES["tinta"])
for k, v in enumerate(comparacion.values):
    ax.annotate(f"{v:.2f}", (v, k), xytext=(4, 0), textcoords="offset points", va="center", fontweight="bold")
ax.set(xlabel="MAE en julio (bicis por hora)", xlim=(1.5, 2.2), title="Menos es mejor")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**¿Qué esperamos ver?** Con las columnas sumadas la recta falla por 2.05 bicis: ni siquiera le gana a la tabla (1.95).
Con la interacción `est_hora_finde` (una columna por estación, hora y tipo de día) más los dos rezagos baja a 1.85.
Son las mismas fuentes de información; lo que cambió es **cómo** se las dimos a la recta.
Una décima por hora parece poco, pero son 40 estaciones por 24 horas por 31 días: unas 3,000 bicis mejor ubicadas en el mes.

### En vivo: escoge columnas

Escoge columnas en la lista (con Ctrl, o Cmd en Mac, escoges varias). Cada combinación se entrena en menos de un segundo
y la barra se compara contra la tabla (1.95). **Antes de correr: ¿`est_hora` sola le gana a la tabla?**

In [ ]:
COLUMNAS = ["departures_lag_24h", "departures_lag_168h", "departures_mean_24h", "departures_mean_168h",
            "hour_of_day", "day_of_week", "is_weekend", "station_id", "est_hora", "est_hora_finde", "est_hora_dia", "capacity"]


def ver_columnas(columnas):
    columnas = list(columnas)
    if not columnas:
        print("Escoge al menos una columna.")
        return
    mae = prueba_columnas(columnas)
    fig, ax = plt.subplots(figsize=(8, 2.4))
    color = COLORES["yhat"] if mae < E_REF else COLORES["error"]
    ax.barh(["tu lista", "la tabla del Lab 1"], [mae, E_REF], color=[color, COLORES["datos"]], edgecolor=COLORES["tinta"])
    for k, v in enumerate([mae, E_REF]):
        ax.annotate(f"{v:.2f}", (v, k), xytext=(4, 0), textcoords="offset points", va="center", fontweight="bold")
    ax.set(xlabel="MAE en julio", xlim=(0, max(3.0, mae + 0.4)))
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    veredicto = "le gana a la tabla" if mae < E_REF else "no le gana a la tabla"
    print(f"MAE en julio: {mae:.3f}, {veredicto}   con {columnas}")

In [ ]:
try:
    import ipywidgets as widgets
    widgets.interact(ver_columnas,
                     columnas=widgets.SelectMultiple(options=COLUMNAS, value=["est_hora"],
                                                     rows=len(COLUMNAS), description="Columnas"))
except ImportError:
    print("No hay ipywidgets: usa la celda de abajo.")

In [ ]:
# Si el widget no aparece: cambia la lista y vuelve a correr esta celda
ver_columnas(["est_hora"])
ver_columnas(["departures_lag_24h", "departures_lag_168h", "est_hora_dia"])

In [ ]:
# Todo lo que llevamos probado, del mejor al peor
display(pd.Series(intentos, name="MAE en julio").sort_values().round(3).to_frame())

**¿Qué esperamos ver?** `est_hora` sola da 2.15: no le gana a la tabla, porque sin rezagos la recta solo aprende el promedio
de cada estación y hora, y además mezcla entre semana con fin de semana. Con los dos rezagos y `est_hora_dia` (estación, hora y día
de la semana) baja a 1.82, la mejor de las que vimos. Esa lista de 6,722 columnas es la que llevamos a la parte 2.

# Parte 2: Ridge

*Se corre en la slide "En Python: Ridge".*

## 4. El problema: una estación nueva con dos semanas

Con nueve meses de datos, cada columna de `est_hora_dia` ve unas 38 filas. Pero piensa en lo que pasa cuando MiBici abre
estaciones nuevas: solo hay dos semanas de historia. Para simularlo, entrenamos con **solo del 17 al 30 de junio**
y seguimos validando con julio.

In [ ]:
dos_semanas = train[train.hour >= "2026-06-17"]          # del 17 al 30 de junio
COLS_RIDGE = ["departures_lag_24h", "departures_lag_168h", "est_hora_dia"]


def arma(alpha):
    """El pipeline de prueba_ridge del Lab 2: one-hot de est_hora_dia y los dos rezagos escalados."""
    columnas = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["est_hora_dia"]),
                                  ("num", StandardScaler(), ["departures_lag_24h", "departures_lag_168h"])])
    return make_pipeline(columnas, LinearRegression() if alpha == 0 else Ridge(alpha=alpha))


def mae(m, datos):
    return np.mean(np.abs(m.predict(datos[COLS_RIDGE]) - datos.departures))


recta0 = arma(0).fit(dos_semanas[COLS_RIDGE], dos_semanas.departures)
nombres = recta0[0].get_feature_names_out()
filas_por_columna = dos_semanas.est_hora_dia.value_counts()
print(f"{len(dos_semanas):,} filas y {len(nombres):,} columnas ({len(nombres) - 2:,} de est_hora_dia y 2 rezagos)")
print(f"cada columna de est_hora_dia tiene {filas_por_columna.min()} filas para aprender su peso")
display(pd.DataFrame({"MAE train": [mae(recta0, dos_semanas)], "MAE julio": [mae(recta0, julio)]},
                     index=["λ = 0 (LinearRegression)"]).round(2))

**¿Qué esperamos ver?** 13,440 filas y 6,722 columnas: casi una columna por cada dos filas. Cada peso de `est_hora_dia` se aprende
con **dos** datos (los dos lunes, los dos martes...). En train el MAE es 1.06 y en julio 2.48: la recta memorizó las dos semanas.
Las dos restas del diagnóstico: train menos la tabla da 1.06 − 1.95 = −0.89 (le gana a la referencia con lo que vio)
y julio menos train da 2.48 − 1.06 = +1.42. Eso es **varianza**.

## 5. La multa de Ridge

$$J(\vec w, b) = \frac{1}{2m}\sum_{i=1}^{m}\big(\hat y^{(i)} - y^{(i)}\big)^2 + \frac{\lambda}{2m}\sum_{j=1}^{n} w_j^2$$

En palabras: el mismo error de la recta, más una multa por el tamaño de los pesos. Cada peso grande paga su cuadrado; $b$ no paga.
$\lambda$ dice cuánto pesa la multa: con $\lambda = 0$ es la recta de siempre, con $\lambda$ enorme todos los pesos se van a cero
y la predicción queda en el promedio.

$$\lVert \vec w \rVert = \sqrt{\textstyle\sum_{j} w_j^2}$$

En palabras: el tamaño del vector de pesos, la raíz de lo que cobra la multa. Lo usamos para ver cuánto encoge Ridge.

**¿Por qué los rezagos se escalan y las columnas de 0 y 1 no?** La multa compara pesos, y el peso depende de la unidad de la columna.
`StandardScaler` pone los rezagos en desviaciones estándar; las columnas de 0 y 1 ya están en una escala comparable.

Ajustamos la rejilla del Lab 2 de una vez (son 7 modelos, unos segundos en total), para que el slider sea instantáneo.

In [ ]:
LAMBDAS = [0, 0.1, 1, 3, 10, 100, 1000]
modelos = {a: (recta0 if a == 0 else arma(a).fit(dos_semanas[COLS_RIDGE], dos_semanas.departures)) for a in LAMBDAS}
pesos = pd.DataFrame({a: m[-1].coef_ for a, m in modelos.items()}, index=nombres)
curva = pd.DataFrame({a: {"MAE train": mae(m, dos_semanas), "MAE julio": mae(m, julio),
                          "‖w‖": np.linalg.norm(m[-1].coef_)} for a, m in modelos.items()}).T
curva.index = pesos.columns = pd.Index(LAMBDAS, dtype=object, name="λ")    # para que 1 se lea 1 y no 1.0
display(curva.round(2).T)

# pesos que vamos a seguir: la estación 51 los lunes de 6 a 10 h y los dos rezagos
ELEGIDOS = {f"cat__est_hora_dia_51_{h}_0": f"51, lun {h} h" for h in range(6, 11)}
ELEGIDOS.update({"num__departures_lag_24h": "rezago 24 h", "num__departures_lag_168h": "rezago 168 h"})
display(pesos.loc[list(ELEGIDOS)].rename(index=ELEGIDOS).round(2))

**¿Qué esperamos ver?** Al subir $\lambda$, el MAE de train sube sin parar (de 1.06 a 2.35) y $\lVert \vec w \rVert$ baja de 525 a 3.
El de julio primero baja (2.48, 2.27, 2.13) y luego vuelve a subir (2.14, 2.17, 2.20, 2.23).
En los pesos, con $\lambda = 0$ el de las 7 h del lunes en la estación 51 vale 38.9 y el rezago de 168 h sale **negativo** (−2.25):
la recta dice que si hace una semana salieron muchas, hoy saldrán pocas. Eso no tiene sentido para el negocio; es la recta
acomodando dos datos por columna. Con $\lambda = 1$ el rezago ya es positivo (1.27).

In [ ]:
def ver_lambda(lam=1):
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [1.6, 1, 1.2]})

    # 1. pesos representativos: λ escogida contra λ = 0
    p0, pl = pesos.loc[list(ELEGIDOS), 0], pesos.loc[list(ELEGIDOS), lam]
    pos = np.arange(len(ELEGIDOS))
    a1.bar(pos - 0.2, p0, width=0.4, color=COLORES["datos"], edgecolor=COLORES["tinta"], label="λ = 0")
    a1.bar(pos + 0.2, pl, width=0.4, color=COLORES["lambda"], edgecolor=COLORES["tinta"], label=f"λ = {lam}")
    a1.axhline(0, color=COLORES["tinta"], lw=1)
    a1.set_xticks(pos, list(ELEGIDOS.values()), rotation=35, ha="right")
    a1.set(ylabel="peso", title="Algunos pesos", ylim=(-5, 42))
    a1.legend()

    # 2. el tamaño de todos los pesos
    colores = [COLORES["lambda"] if a == lam else COLORES["datos"] for a in LAMBDAS]
    a2.bar([str(a) for a in LAMBDAS], curva["‖w‖"], color=colores, edgecolor=COLORES["tinta"])
    a2.set(yscale="log", xlabel="λ", ylabel="‖w‖ (escala log)", title=f"‖w‖ = {curva.loc[lam, '‖w‖']:.1f}")

    # 3. la U de julio con el punto marcado
    x = np.arange(len(LAMBDAS))
    k = LAMBDAS.index(lam)
    a3.plot(x, curva["MAE train"], marker="o", color=COLORES["w"], label="train (dos semanas)")
    a3.plot(x, curva["MAE julio"], marker="o", color=COLORES["error"], label="julio")
    a3.axhline(E_REF, color=COLORES["tinta"], lw=1, ls=":", label="la tabla (1.95)")
    a3.scatter([k, k], curva.loc[lam, ["MAE train", "MAE julio"]], s=180, facecolors="none",
               edgecolors=COLORES["lambda"], linewidths=2.5, zorder=4)
    a3.set_xticks(x, [str(a) for a in LAMBDAS])
    a3.set(xlabel="λ", ylabel="MAE", ylim=(0.9, 2.6),
           title=f"train {curva.loc[lam, 'MAE train']:.2f}   julio {curva.loc[lam, 'MAE julio']:.2f}")
    a3.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
try:
    import ipywidgets as widgets
    widgets.interact(ver_lambda, lam=widgets.SelectionSlider(options=LAMBDAS, value=1, description="λ"))
except ImportError:
    print("No hay ipywidgets: usa la celda de abajo.")

In [ ]:
# Si el widget no aparece: cambia λ (0, 0.1, 1, 3, 10, 100 o 1000) y vuelve a correr esta celda
ver_lambda(1)

**¿Qué esperamos ver?** Recorramos el slider de izquierda a derecha.
Con $\lambda = 0$ las barras naranjas son iguales a las grises y $\lVert \vec w \rVert$ = 525.
Con $\lambda = 1$ los pesos de la estación 51 se encogen a menos de la mitad (el de las 7 h, de 38.9 a 15.3), el rezago de 168 h
cambia de signo, $\lVert \vec w \rVert$ baja a 160 y julio llega a su mínimo, 2.13.
Con $\lambda = 1000$ los pesos de la estación casi desaparecen y solo quedan los rezagos: train y julio se juntan alrededor de 2.2 a 2.3,
los dos peor que la tabla. Eso es **sesgo**: la multa ya no deja aprender nada propio de cada estación.

**¿Por qué los rezagos encogen menos?** Cada rezago se aprende con las 13,440 filas; cada peso de `est_hora_dia` con dos.
Ridge cobra igual a todos, pero a un peso respaldado por muchos datos le conviene más pagar la multa.

## 6. Escoger λ con julio

$$\lambda^\ast = \arg\min_{\lambda} \ \text{MAE}_{\text{julio}}(\lambda)$$

En palabras: escogemos la $\lambda$ de la rejilla que menos se equivoca en julio, el mes que no se usó para ajustar.

**¿Por qué no con train?** Porque en train siempre gana $\lambda = 0$: sin multa, la recta tiene toda la libertad de acomodarse a lo que ve.

In [ ]:
mejor = curva["MAE julio"].idxmin()
print(f"λ escogida con julio: {mejor}   MAE en julio {curva.loc[mejor, 'MAE julio']:.2f} "
      f"(contra {curva.loc[0, 'MAE julio']:.2f} sin multa)")
print(f"Con train habríamos escogido λ = {curva['MAE train'].idxmin()}")

# el ejemplo de la estación 51 los lunes a las 8 a.m.
lunes8 = julio[(julio.station_id == 51) & (julio.day_of_week == 0) & (julio.hour_of_day == 8)]
ejemplo = pd.DataFrame({"real": lunes8.departures.to_numpy(),
                        "λ = 0": modelos[0].predict(lunes8[COLS_RIDGE]),
                        f"λ = {mejor}": modelos[mejor].predict(lunes8[COLS_RIDGE]),
                        "hace una semana": lunes8.departures_lag_168h.to_numpy()},
                       index=lunes8.hour.dt.strftime("lun %d jul"))
display(ejemplo.round(0).astype(int))
visto = dos_semanas[(dos_semanas.station_id == 51) & (dos_semanas.day_of_week == 0) & (dos_semanas.hour_of_day == 8)]
print("Lo único que vio el modelo de esa columna:", dict(zip(visto.hour.dt.strftime("lun %d jun"), visto.departures)))

fig, ax = plt.subplots(figsize=(9, 3.8))
pos = np.arange(len(ejemplo))
for k, (col, color) in enumerate([("real", COLORES["tinta"]), ("λ = 0", COLORES["datos"]), (f"λ = {mejor}", COLORES["lambda"])]):
    ax.bar(pos + (k - 1) * 0.27, ejemplo[col], width=0.27, color=color, edgecolor=COLORES["tinta"], label=col)
    for x, v in zip(pos + (k - 1) * 0.27, ejemplo[col]):
        ax.annotate(f"{v:.0f}", (x, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
ax.set_xticks(pos, ejemplo.index)
ax.set(ylabel="salidas a las 8 a.m.", title="Estación 51, lunes a las 8 a.m. de julio")
ax.legend()
plt.tight_layout()
plt.show()

**¿Qué esperamos ver?** Julio escoge $\lambda = 1$ con MAE de 2.13; train habría escogido $\lambda = 0$.
En la estación 51 los lunes a las 8 a.m. salieron 62, 65, 32 y 34 bicis. Sin multa la recta predijo 12, 2, 1 y 15:
el 13 de julio, cuando hace una semana habían salido 62, predijo 2, por el peso negativo del rezago de 168 h.
Con $\lambda = 1$ predijo 24, 31, 34 y 24. Todavía se queda corta los dos primeros lunes, pero ya no manda un camión vacío.
El modelo solo había visto dos lunes de esa estación a esa hora (11 y 37 bicis): con eso, cualquier peso es una apuesta.

## 7. ¿Cuándo sirve Ridge?

Misma rejilla, mismas columnas, pero ahora con **todo train** (254 mil filas): cada columna de `est_hora_dia` ve unas 38 filas
en lugar de dos. Son 7 modelos más; tarda unos segundos.

In [ ]:
modelos_todo = {a: arma(a).fit(train[COLS_RIDGE], train.departures) for a in LAMBDAS}
julio_todo = pd.Series([mae(modelos_todo[a], julio) for a in LAMBDAS], index=curva.index)
lado = pd.DataFrame({"dos semanas (13,440 filas)": curva["MAE julio"], "todo train (254,440 filas)": julio_todo})
lado.index.name = "λ"
display(lado.round(2).T)

fig, ax = plt.subplots(figsize=(10, 4))
pos = np.arange(len(LAMBDAS))
ax.bar(pos - 0.2, lado.iloc[:, 0], width=0.4, color=COLORES["lambda"], edgecolor=COLORES["tinta"], label=lado.columns[0])
ax.bar(pos + 0.2, lado.iloc[:, 1], width=0.4, color=COLORES["w"], edgecolor=COLORES["tinta"], label=lado.columns[1])
ax.axhline(E_REF, color=COLORES["tinta"], lw=1, ls=":", label="la tabla (1.95)")
ax.set_xticks(pos, [str(a) for a in LAMBDAS])
ax.set(xlabel="λ", ylabel="MAE en julio", ylim=(1.6, 2.6), title="Ridge ayuda con pocos datos y estorba con muchos")
ax.legend(loc="upper center", ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

**¿Qué esperamos ver?** Con dos semanas (naranja) julio dibuja la U: baja de 2.48 a 2.13 en $\lambda = 1$ y vuelve a subir.
Con todo train (azul) no hay U: el mejor es $\lambda = 0$ con 1.82, y al subir $\lambda$ el MAE solo crece, hasta 2.12 con $\lambda = 1000$.
Con 38 filas por columna el promedio ya es confiable y no hay ruido que frenar; la multa solo mete sesgo.
Además, las barras azules quedan por debajo de las naranjas en toda la rejilla: **más datos le ganan a cualquier $\lambda$**.
Ridge es el remedio para la varianza, no una mejora que se le pone a todo.

## Tus preguntas y reflexiones

No se entregan. Escribe dos o tres líneas en cada una; las retomamos en el lab.

**1. MiBici abre 10 estaciones en Zapopan el próximo mes.** Tendrás dos semanas de datos de cada una cuando te pidan el plan
de redistribución. ¿Usarías Ridge? ¿Con qué $\lambda$ empezarías y cómo sabrías cuándo dejar de usarlo?
Piensa también en qué columnas: ¿`est_hora_dia` o `est_hora_finde`?

*Tip: compara las dos curvas de la sección 7 y cuenta cuántas filas tendría cada columna con cada opción.*

_Escribe aquí._

**2. El lunes 13 de julio la recta sin multa predijo 2 bicis y salieron 65.** Si redistribución hubiera creído ese número,
¿qué les pasa a los usuarios de la estación 51 esa mañana? ¿Qué le dirías al gerente que pregunta por qué el modelo "se volvió loco"?

*Tip: vuelve a la tabla de pesos con $\lambda = 0$ y fíjate en el signo del rezago de 168 h.*

_Escribe aquí._

**3. Te piden una columna nueva para el modelo.** El equipo propone: la lluvia de esa hora, si es día festivo, las llegadas
de esa hora (`arrivals`) y si hay un partido en el estadio. ¿Cuáles se pueden usar si decidimos con un día de anticipación?
¿Cuál inventarías primero y de qué tipo sería (rezago, categoría o interacción)?

*Tip: para cada una, pregúntate si a las 8 a.m. de hoy ya conoces su valor para las 8 a.m. de mañana.*

_Escribe aquí._